# LoRA / QLoRA Customer Support Fine-Tuning

This notebook prepares the data, trains the adapter, and evaluates the model inside Kaggle.

The workflow writes all generated artifacts to `/kaggle/working/` so the outputs persist after the session.

In [ ]:
from pathlib import Path

repo_root = Path('/content/Finetund-custumer-support-')
if not (repo_root / 'scripts' / 'prepare_data.py').exists():
    repo_root = Path.cwd()

!pip install -r {repo_root / 'requirements.txt'}


## Load the dataset from Colab storage

This cell looks through `/content/` for the Bitext CSV and copies it into the repo's `data/raw/` folder.


In [ ]:
import shutil

input_root = Path('/content')
raw_dir = repo_root / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(input_root.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError('No CSV file found under /content/')

source_csv = csv_files[0]
target_csv = raw_dir / source_csv.name
shutil.copy2(source_csv, target_csv)
print(f'Copied {source_csv} -> {target_csv}')


## Prepare the dataset

The preparation step converts the CSV into instruction-format JSONL files and builds the IID/OOD evaluation splits.

In [ ]:
import subprocess

subprocess.run([
    'python3',
    str(repo_root / 'scripts' / 'prepare_data.py'),
    '--raw-dir',
    str(repo_root / 'data' / 'raw'),
    '--output-dir',
    str(repo_root / 'data' / 'processed'),
], check=True)


## Train the adapter

This runs QLoRA training with the YAML config in the repository.

In [ ]:
import subprocess

subprocess.run([
    'python3',
    str(repo_root / 'scripts' / 'train.py'),
    '--config',
    str(repo_root / 'configs' / 'qlora_config.yaml'),
    '--data-dir',
    str(repo_root / 'data' / 'processed'),
    '--output-dir',
    str(repo_root / 'outputs'),
], check=True)


## Evaluate the adapter

The evaluation script compares the fine-tuned adapter with the base model on IID and OOD examples, then writes a JSON summary.

In [ ]:
import subprocess

subprocess.run([
    'python3',
    str(repo_root / 'scripts' / 'evaluate.py'),
    '--config',
    str(repo_root / 'configs' / 'qlora_config.yaml'),
    '--data-dir',
    str(repo_root / 'data' / 'processed' / 'eval'),
    '--adapter-dir',
    str(repo_root / 'outputs' / 'final_adapter'),
    '--output-file',
    str(repo_root / 'outputs' / 'eval_results.json'),
], check=True)


## Outputs

The trained adapter, training logs, and evaluation summary are written to `/content/Finetund-custumer-support-/outputs/`.
